# 🎙️ ELEVEN AUDIO — Estúdio de Voz Profissional (Google Colab GPU)
### Clonagem de Voz, Síntese Neural, Transcrição, Tradução e Dublagem
- **Base:** VoiceStudio (AGPL-3.0) + Eleven Audio (Gabriel Kendy)
- **Persistência Total:** Google Drive (`/content/drive/MyDrive/ElevenAudio`) — modelos e áudios salvos permanentemente.
- **Resiliência:** Tratamento anti-erro para Cloudflare Tunnel e auto-perfil para OmniVoice.

### 1. Verificação de Aceleração por GPU

In [ ]:
import subprocess
try:
    gpu_info = subprocess.check_output(['nvidia-smi'], text=True)
    print(gpu_info)
    print("\n✅ GPU detectada com sucesso! Pronto para geração ultra-rápida.")
except Exception as e:
    print("⚠️ ATENÇÃO: Nenhuma GPU NVIDIA detectada!")
    print("Vá no menu superior: 'Ambiente de execução' -> 'Alterar tipo de ambiente de execução' -> Selecione 'T4 GPU' e salve.")

### 2. Montar Google Drive & Pastas de Persistência

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/ElevenAudio'
CACHE_HF = os.path.join(DRIVE_BASE, 'cache_hf')
CACHE_TORCH = os.path.join(DRIVE_BASE, 'cache_torch')
PASTA_SAIDAS = os.path.join(DRIVE_BASE, 'saidas')
PASTA_DADOS = os.path.join(DRIVE_BASE, 'dados')

for p in [CACHE_HF, CACHE_TORCH, PASTA_SAIDAS, PASTA_DADOS]:
    os.makedirs(p, exist_ok=True)

os.environ['HF_HOME'] = CACHE_HF
os.environ['TRANSFORMERS_CACHE'] = CACHE_HF
os.environ['TORCH_HOME'] = CACHE_TORCH
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

print("✅ Google Drive conectado!")
print(f"📁 Seus modelos e áudios ficam salvos em: {DRIVE_BASE}")

### 3. Instalação & Patch Anti-Erro (Cloudflare & OmniVoice)

In [ ]:
# 1. Instalação de utilitários do sistema
!apt-get update -qq && apt-get install -y -qq ffmpeg
!pip install -q uv fastapi uvicorn httpx python-multipart pydub

%cd /content

# 2. Baixa Eleven Audio se não existir
import urllib.request, zipfile, os

zip_url = 'https://github.com/gabrielkendy/eleven-audio/releases/latest/download/eleven-audio.zip'
zip_path = '/content/eleven-audio.zip'
if not os.path.exists('/content/eleven-audio'):
    print("Baixando pacote Eleven Audio...")
    urllib.request.urlretrieve(zip_url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content')
    print("✅ Eleven Audio descompactado!")

# 3. Clona VoiceStudio se não existir
if not os.path.exists('/content/VoiceStudio'):
    print("Clonando motor VoiceStudio...")
    !git clone --depth 1 https://github.com/debpalash/VoiceStudio.git /content/VoiceStudio
    print("✅ VoiceStudio clonado!")

%cd /content/VoiceStudio
!pip install -q -r requirements.txt || pip install -q TTS soundfile librosa
%cd /content/eleven-audio
!pip install -q -r requirements.txt

# 4. PATCH: Corrige api() no gerar.js para nunca estourar SyntaxError com HTML do Cloudflare
gerar_js_path = '/content/eleven-audio/web/gerar.js'
if os.path.exists(gerar_js_path):
    with open(gerar_js_path, 'r', encoding='utf-8') as f:
        js = f.read()
    old_api = 'const corpo = await resposta.json();'
    new_api = '''let corpo = null;
      const cType = resposta.headers.get("content-type") || "";
      if (cType.includes("application/json")) {
        try { corpo = await resposta.json(); } catch(_) {}
      }
      if (!resposta.ok) {
        if (corpo && corpo.detail) throw new Error(corpo.detail);
        if (resposta.status === 502 || resposta.status === 504) throw new Error("O motor recusou ou demorou para responder. Se estiver usando OmniVoice, selecione uma voz clonada na aba 'Clonar'.");
        throw new Error(`Erro ${resposta.status}: Não foi possível concluir.`);
      }'''
    if old_api in js:
        js = js.replace(old_api, new_api)
        with open(gerar_js_path, 'w', encoding='utf-8') as f:
            f.write(js)
        print("✅ Patch aplicado em web/gerar.js (anti-erro JSON)!")

# 5. PATCH: Atualiza rotas.py para retornar 422 em vez de 502 (evita sequestro pelo Cloudflare)
rotas_py_path = '/content/eleven-audio/app/rotas.py'
if os.path.exists(rotas_py_path):
    with open(rotas_py_path, 'r', encoding='utf-8') as f:
        py = f.read()
    old_502 = 'raise HTTPException(502, f"erro da base: {erro}") from erro'
    new_422 = 'raise HTTPException(422, f"Aviso do motor: {erro}. Dica: Para OmniVoice, crie e selecione uma voz clonada na aba Clonar.") from erro'
    if old_502 in py:
        py = py.replace(old_502, new_422)
        with open(rotas_py_path, 'w', encoding='utf-8') as f:
            f.write(py)
        print("✅ Patch aplicado em app/rotas.py (anti-502 Cloudflare)!")

print("\n🚀 Preparação concluída!")

### 4. Inicialização dos Serviços & Túnel Cloudflare

In [ ]:
import subprocess, time, re, os

# Baixa o binário do cloudflared
if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /usr/local/bin/cloudflared

# Inicia o VoiceStudio backend (porta 3900)
print("Inicializando motor VoiceStudio (Porta 3900)...")
p_base = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'backend.main:app', '--host', '127.0.0.1', '--port', '3900'],
    cwd='/content/VoiceStudio',
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Configura persistência no Drive
os.environ['ELEVEN_SAIDAS'] = '/content/drive/MyDrive/ElevenAudio/saidas'
os.environ['ELEVEN_DADOS'] = '/content/drive/MyDrive/ElevenAudio/dados'

# Inicia o Eleven Audio Studio (porta 7800)
print("Inicializando interface Eleven Audio (Porta 7800)...")
p_app = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'app.servidor:app', '--host', '127.0.0.1', '--port', '7800'],
    cwd='/content/eleven-audio',
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

# Inicia o túnel Cloudflare para a porta 7800
print("Abrindo túnel Cloudflare seguro...")
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7800'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for line in tunnel_proc.stdout:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("\n" + "="*70)
    print(f"🎉 ELEVEN AUDIO NO AR COM SUCESSO!")
    print(f"👉 ACESSE SEU ESTÚDIO AQUI: {public_url}")
    print("="*70 + "\n")
    print("💡 DICA PARA O MOTOR OMNIVOICE:")
    print("1. O OmniVoice é um clonador neural. Antes de gerar, vá na aba 'Clonar'.")
    print("2. Envie um áudio curto (10s a 30s) com uma voz limpa e salve o perfil.")
    print("3. Na aba 'Compor', selecione essa voz clonada e clique em 'Gerar áudio'!")
else:
    print("⚠️ Link do túnel não obtido automaticamente.")

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    p_base.terminate()
    p_app.terminate()
    tunnel_proc.terminate()